# Notebook 13 – Preventing Data Leakage
## What is Data Leakage?

Data leakage happens when information that shouldn't be available at
prediction time accidentally makes its way into the training process. The
model then performs suspiciously well during testing, but falls apart the
moment it faces genuinely new, real-world data, because it learned to rely
on information it will never actually have in production.



In [2]:
import pandas as pd

df = pd.read_csv("hr_employee_attrition_raw.csv")
df["Attrition_binary"] = df["Attrition"].map({"Yes": 1, "No": 0})
df["MonthlyIncome_clean"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False), errors="coerce"
)
df.shape

(1230, 18)

## Target Leakage

**What it is:** using a feature that only exists, or only has meaningful
values, because of the outcome you're trying to predict.

Our dataset doesn't contain a naturally leaky column, so let's build one
on purpose to see exactly what this looks like and why it's dangerous.
Imagine HR added a column called `ExitInterviewCompleted`, only filled in
for employees who actually left. That column wouldn't exist yet for a
current employee you're trying to predict attrition for, so using it in
training creates a feature the model will never have access to when it
actually matters.

In [3]:
# INCORRECT: this feature only exists because the person already left
df["ExitInterviewCompleted_LEAKY"] = df["Attrition"].apply(lambda x: 1 if x == "Yes" else 0)

correlation = df["ExitInterviewCompleted_LEAKY"].corr(df["Attrition_binary"])
print("Correlation with target:", correlation)

Correlation with target: 1.0


In [4]:
# CORRECT: drop any feature that couldn't exist before the outcome happened
df_correct = df.drop(columns=["ExitInterviewCompleted_LEAKY"])



## Train-Test Contamination

**What it is:** fitting any preprocessing step (scaler, encoder, imputer)
on the full dataset before splitting, so statistics from the test set leak
into training. We already hit this exact mistake in Notebook 12.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# INCORRECT: fit on the whole dataset, then split
scaler_wrong = StandardScaler()
df["Income_scaled_WRONG"] = scaler_wrong.fit_transform(df[["MonthlyIncome_clean"]].fillna(0))
train_wrong, test_wrong = train_test_split(df, test_size=0.2, random_state=42)
print("Mean used includes rows that ended up in test:", scaler_wrong.mean_)

Mean used includes rows that ended up in test: [5498.18313781]


In [6]:
# CORRECT: split first, fit only on training data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Attrition"])

scaler_correct = StandardScaler()
train_df = train_df.copy()
test_df = test_df.copy()
train_df["Income_scaled"] = scaler_correct.fit_transform(train_df[["MonthlyIncome_clean"]].fillna(0))
test_df["Income_scaled"] = scaler_correct.transform(test_df[["MonthlyIncome_clean"]].fillna(0))

print("Mean used, training data only:", scaler_correct.mean_)

Mean used, training data only: [5369.52788129]


## Feature Leakage

**What it is:** a feature that indirectly encodes the target through a
roundabout path, even if it's not an obvious duplicate of the label.

Example: suppose we engineered a feature called `AverageDeptAttritionRate`
by grouping the **entire dataset** by department and computing the
attrition rate, then attaching that rate back to every row, including the
row's own outcome baked into its own group average. Each employee's row
technically influenced the very feature describing their own risk.

In [7]:
# INCORRECT: each row's own outcome is included in the average used to describe it
df["DeptAttritionRate_LEAKY"] = df.groupby("Department")["Attrition_binary"].transform("mean")

# This "feature" is calculated using every row's own true label,
# an employee who left drags their own department's average upward,
# which then gets used to help "predict" that very employee's outcome

In [8]:
# CORRECT: compute the average using only training rows, excluding the row itself when possible,
# and apply the same fitted average to test data without recomputing
dept_attrition_rate = train_df.groupby("Department")["Attrition_binary"].mean()

train_df["DeptAttritionRate"] = train_df["Department"].map(dept_attrition_rate)
test_df["DeptAttritionRate"] = test_df["Department"].map(dept_attrition_rate)  # apply, don't refit

## Preprocessing Leakage

**What it is:** the same underlying mistake as train-test contamination,
but specifically about steps like imputation, outlier bounds, or encoding,
not just scaling. We saw this exact issue with target encoding back in
Notebook 7, when `JobRole_target_encoded` was accidentally computed before
the train-test split.

In [9]:
# INCORRECT: imputing with the full dataset's median before splitting
full_median = df["MonthlyIncome_clean"].median()
df["Income_imputed_WRONG"] = df["MonthlyIncome_clean"].fillna(full_median)

In [10]:
# CORRECT: compute the median from training data only, apply it everywhere
train_median = train_df["MonthlyIncome_clean"].median()

train_df["Income_imputed"] = train_df["MonthlyIncome_clean"].fillna(train_median)
test_df["Income_imputed"] = test_df["MonthlyIncome_clean"].fillna(train_median)  # same value, not recomputed

## Temporal Leakage

**What it is:** using information from the future to predict something in
the past, common when a dataset has a time component like our `JoinDate`.

Example: imagine training a model on data through 2025, but the training
process randomly puts a 2015 hire's row in the test set and a 2024 hire's
row in training. The 2024 hire's outcome might already reflect market
conditions or company policy changes that hadn't happened yet back when
the 2015 hire's outcome was determined, an unrealistic mix of time periods.

In [11]:
# INCORRECT: random split ignores the timeline entirely
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")
train_random, test_random = train_test_split(df, test_size=0.2, random_state=42)
print("Random split mixes years:")
print("Train date range:", train_random["JoinDate"].min(), "-", train_random["JoinDate"].max())
print("Test date range:", test_random["JoinDate"].min(), "-", test_random["JoinDate"].max())

Random split mixes years:
Train date range: 2015-01-01 00:00:00 - 2025-05-26 00:00:00
Test date range: 2015-01-04 00:00:00 - 2025-03-23 00:00:00


In [12]:
# CORRECT: split by time, older data trains, newer data tests
df_sorted = df.sort_values("JoinDate")
cutoff = int(len(df_sorted) * 0.8)

train_temporal = df_sorted.iloc[:cutoff]
test_temporal = df_sorted.iloc[cutoff:]

print("Train date range:", train_temporal["JoinDate"].min(), "-", train_temporal["JoinDate"].max())
print("Test date range:", test_temporal["JoinDate"].min(), "-", test_temporal["JoinDate"].max())

Train date range: 2015-01-01 00:00:00 - 2023-02-24 00:00:00
Test date range: 2023-03-02 00:00:00 - 2025-05-26 00:00:00


## How to Detect Leakage

A few warning signs worth checking every time a model performs unusually well:

* **Suspiciously high accuracy or F1-score**, especially on a hard,
  imbalanced problem like our 81/19 attrition split. If accuracy jumps to
  98%+, be suspicious before being impressed.
* **One feature dominates feature importance** far more than the rest,
  worth investigating whether that feature could only exist because of
  the outcome.
* **Performance drops sharply on genuinely new data** after deployment,
  compared to what validation scores predicted, a classic leakage symptom
  that only shows up after the fact.
* **A feature has a near-perfect correlation with the target**, like our
  intentionally leaky `ExitInterviewCompleted_LEAKY` example above.

In [13]:
# Quick leakage check: does any feature correlate suspiciously highly with the target?
numeric_cols = df.select_dtypes(include="number").columns
suspicious = df[numeric_cols].corr()["Attrition_binary"].drop("Attrition_binary").abs().sort_values(ascending=False)
print(suspicious.head())

ExitInterviewCompleted_LEAKY    1.000000
DeptAttritionRate_LEAKY         0.067558
Age                             0.025533
Income_scaled_WRONG             0.020184
RandomSurveyCode                0.017647
Name: Attrition_binary, dtype: float64


## How to Prevent Leakage

A simple checklist, reused from Notebook 12 and expanded here:

* Split the data first, always, before any `.fit()` or `.fit_transform()` call.
* For every feature, ask: "would this value actually be available at the
  moment I need to make a prediction?" If not, drop it.
* If the data has a time dimension, split by time instead of randomly.
* Be suspicious of features that are derived from group-level statistics
  (like department averages) if the target label of the current row
  contributed to that statistic.
* Treat unusually high performance as a signal to investigate, not
  celebrate, before trusting it.